# Anlise MPS vs Pipeline Interno (2 Instância TJRJ)

**Objetivo**: Apresentar as diferenças encontradas entre o processo de tranformação da MPS (responsásvel pelo envio ao CNJ/DataJud) e nosso pipeline interno (`2inst_situacao` + `2inst_metacnj`).

**Contexto**: A MPS opera o banco Oracle do EJUD e é responsável por:
1. Traduzir movimentos internos EJUD  códigos CNJ (via `BUSCARDADOS_FASECOMPL_CNJ`)
2. Preencher a tabela `TAB_XML_ANDAMENTO` (pré computada, já com códigos CNJ)
3. Calcular metas CNJ a partir dessa tabela (`METAS_CNJ_DATAJUD_2024_GERA`)
4. Enviar XML ao DataJud

Nós replicamos os passos 1-3 de forma independente no AWS (PySpark + Iceberg).

---

## 1. Resolução de Órgão Julgador (cod_oj)

### Como a MPS faz (BuscarOrgaoJulgadorMovimento)

```
MovimentoRelator.CodOrgJulg
   LEFT JOIN CamaraCivelEspecializada (CodOrgJulgCiv  CodOrgJulgEspec)
   JOIN OrgaoJulgador.CodOrgJulgCNJ
```

**Filtros adicionais MPS:**
- Exclui distribuies canceladas (`MovCanceDistr`)
- Cadeia de fallback: última dist antes do mov  primeira dist depois
- Se OJ = NULL  movimento **não é enviado ao CNJ**

### Como é feito (`_enrich_relator`)

```
ejud.movimentorelator.codorgjulg  direto (último movimento  cutoff)
   anti-join ejud.movcancedistr (filtro de segurança)
```

### Diferenças

| Aspecto | MPS | Nós |
|---------|-----|----|
| Tradução CamaraCivelEspecializada | Sim (27 linhas, mapeia câmaras cíveis antigas  especializadas) | Não |
| Tradução OrgaoJulgador.CodOrgJulgCNJ | Sim | Não |
| Filtro distribuição cancelada | Sim (`MovCanceDistr`) | **Sim** (`ejud.movcancedistr`) |
| Fallback cadeia de distribuição | Sim (3 níveis) | Não  usa último movimento direto |

### Decisão tomada

**não aplicamos** `CamaraCivelEspecializada` nem tradução para `CodOrgJulgCNJ`.

**Motivo**: Usar códigos internos brutos nãp parece consistente nos dados-fonte. Se um OJ estiver errado na origem, nós **vemos o problema** e podemos agir. A MPS traduz silenciosamente para um código CNJ válido, **mascarando** o erro. select * from ejud.CamaraCivelEspecializada@tj01

> Ref: `iceberg/2inst_datajud/mps_procedure_analysis.md`  seo "Análise das procedures do cálculo da Meta"

## 2. Tradução de Movimentos  códigos CNJ

### Como a MPS faz (BUSCARDADOS_FASECOMPL_CNJ)

Funo Oracle com fallback hierrquico em 4 nveis:

```
Nvel 5: codfase + faseexterna + codcompl (mais especfico)
Nvel 4: codfase + faseexterna (sem complemento)
Nvel 3: codfase + codcompl
Nvel 2: codfase sozinho (menos especfico)
```

Se **nenhum nível** retorna resultado  movimento **não  enviado**.

### Como nós fazemos (2inst_situacao)

Tabela `integracaofasecompl_ejud_cnj`  mesma lógica de fallback hierárquico.
Movimentos sem tradução CNJ  **não geram situação**  portanto no entram na meta.

### Status: **EQUIVALENTE**

Ambos usam a mesma cadeia. Diferena  que MPS roda no Oracle e nós no Spark.

> Ref: `iceberg/2inst_datajud/mps_procedure_analysis.md`  seo "Análise das procedures do cálculo da Meta"

## 3. Resolução de Classe (cod_classe)

### Como a MPS faz (BuscarClasseCNJMudancaClasse)

```
Se existe MovMudClass (mudança de classe) ANTES do movimento:
   usar TipoProcesso.CodExtCNJ da nova classe
Seno:
   usar classe original do processo (TipoProcesso.CodExtCNJ)
```

### Como nós fazemos

Pipeline `2inst_situacao` já resolve a classe CNJ via `ejud.tipoprocesso.codextcnj`.
Meta herda de situacao.

### Status: **EQUIVALENTE**

Ambos usam `TipoProcesso.CodExtCNJ`. Mudança de classe  coberta pelo pipeline de situação.

## 4. Distribuições Canceladas

### Como a MPS faz

Tabela `MovCanceDistr`  exclui movimentos cuja distribuição foi cancelada.
Usado em `BuscarOrgaoJulgadorMovimento` para não considerar distribuição inválidas.

### Como nós fazemos

Anti-join `ejud.movcancedistr` no `_enrich_relator` (aplicado em m01, m02, mNN).
Mesmo padro já usado no script `relatorio_resp`.

### Status: **EQUIVALENTE** (aps refatoração)

> Implementado em: `ejud_metacnj_m01_emr.py`, `ejud_metacnj_m02_emr.py`, `ejud_metacnj_mNN_emr.py`

## 5. TAB_XML_ANDAMENTO (MPS) vs 2inst_situacao (nós)

### O que  TAB_XML_ANDAMENTO

Tabela Oracle pr-computada pela MPS:
- Atualizada diariamente (`ATUALIZA_DIARIO_XML_ANDAMENTO`) + rebuild menósal
- Cada linha = 1 movimento j traduzido para CNJ
- Campos: `CodDoc`, `DtHrMov`, `Cod_Tip_Mov_Num` (CNJ), `Cod_OJ`, `Cod_Classe`, complementos

### O que  nosso 2inst_situacao

Tabela Iceberg `dw_2inst_ouro.tb_tf_situacao_ejud`:
- Cada linha = 1 movimento traduzido para situação CNJ
- Campos: `coddoc`, `dthrmov`, `cod_tip_mov` (CNJ), `situacao`, `cod_oj`, `cod_classe`

### Analogia direta

| TAB_XML_ANDAMENTO (MPS/Oracle) | tb_tf_situacao_ejud (nós/Iceberg) |
|-------------------------------|-----------------------------------|
| Cod_Tip_Mov_Num | cod_tip_mov |
| Cod_OJ | cod_oj |
| Cod_Classe | cod_classe |
| DtHrMov | dthrmov |
| CodDoc | coddoc |

**nossa tabela vai almém**: inclui `situacao` (agrupamento CNJ parametrizado) que a MPS não calcula nesse nível.

> Ref: `iceberg/2inst_datajud/mps_procedure_analysis.md`  seo "Insights para arquitetura"

## 6. Filtros de Exclusão MPS vs Nossos

| Filtro | MPS | nós | Status |
|--------|-----|-----|--------|
| `Cod_OJ IS NOT NULL` | Exclui movimento | Não exclui (mantem com OJ vazio) | **DIFERENÇA INTENCIONAL** |
| `CodFaseExt NOT IN (50999,60009,60010,60022)` | Exclui | 2inst_situacao já trata | OK |
| `IndFaseInvis = 'N'` | Exclui invisveis | 2inst_situacao já filtra | OK |
| `BUSCARDADOS_FASECOMPL_CNJ() IS NOT NULL` | Sem tradução = excluído | Sem tradução = sem situação = excluído | OK |
| `codcompt NOT IN (9,10,11,14,15,19,20,22)` | Exclui competencias | nós: `competencia_excluída: [9, 12, 21]` | **DIFERENÇA** |

### Diferenças explicadas

1. **OJ NULL**: MPS não envia ao CNJ. nós mantemos para visibilidade  se processo tem movimento sem OJ, queremos ver.

2. **Competencias**: MPS exclui 8 competencias (Meta): `9,10,11,14,15,19,20,22`. nós excluímos 3: `9, 12, 21`.

## 7. Perdas Identificadas e Tratadas

Auditoria completa de 10 queries Oracle identificou movimentos EJUD sem tradução CNJ.

| # | Descrição | Volume | Ação |
|---|----------|--------|----|
| 5 | Decisão em Petição (codfase=29 => faseext=60006) | 6.434 | *Alterar para* faseext 600063 (Decisão) |
| 6 | Despacho em Petição (codfase=30 => faseext=60005) | 52.336 | *Alterar para* faseext 60005 e 11009 (Despacho) |
| 9 | Despacho codcompl=60438 (Aud. Mediação) | 91 | *Alterar para* Despacho base |

**Total recuperado**: ~59.000 movimentos que antes caíam no vazio.

**Ignorados** (dados antigos/void): Baixa provisória (1.651), Remessa Distribuidor (339), Baixa diligência (137), Pedido Desarquivamento (54.946  requesição interna, CNJ 893).

> Ref: `iceberg/2inst_datajud/mps_procedure_analysis.md`  seo "Validação das perda"

## 8. Resumo de Decisões Arquiteturais

| Decisão | Motivo |
|--------|--------|
| Não traduzir OJ via CamaraCivelEspecializada | Expor problemas de dados em vez de mascarar |
| Manter movimentos com OJ vazio | Visibilidade  podemos comparar e identificar gaps |
| Filtrar distribuição canceladas | Alinhamento com MPS (boas práticas) |
| Replicar fallback hierárquico de tradução | Garantir cobertura equivalente ao MPS |
| Pipeline independente (no depender da TAB_XML_ANDAMENTO) | Autonomia, auditabilidade, detecção de erros MPS |

---

## 9. Próximos Passos

1. **Validação cruzada**: Comparar nosso resultado com DataJud (situação atual publicada pelo CNJ)
2. **Flags de elegibilidade**: Adicionar `fl_elegivel_datajud` para produzir dois resultados
3. **Reunião com 1inst**: Alinhar uso das tabelas de vigência e modelo canônico
4. **Apresentação a diretores**: Resultado = X (nós) vs Y (MPS)  quem decide  o diretor

> Documentação completa: `iceberg/2inst_datajud/`